# Face expression recognition

Este notebook implementa uma CNN baseada na AnyNet para reconhecimento de expressões faciais: _qual é a expressão dessa pessoa?_

---
## Objetivo: 
Classificar imagens faciais em uma das 8 expressões:
| Classe | Expressão |
|--------|-----------|
|    0   |   Raiva   |
|    1   |   Nojo    |
|    2   |   Medo    |
|    3   |Felicidade |
|    4   | Tristeza  |
|    5   | Surpresa  |
|    6   |   Neutro  |
|    7   | Desprezo  |

---

In [1]:
# fer_anynet_optuna.py
# Trabalho Final — Visão Computacional
# CNN baseada em AnyNet para Reconhecimento de Expressões Faciais
# + Otimização de hiperparâmetros com Optuna + Checkpointing (ambiente Kaggle)

import os
import copy
import time

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torchvision.datasets import ImageFolder

import optuna
from optuna.trial import TrialState


In [2]:
# =============================================================
# 0. CONFIGURAÇÃO (ajuste aqui conforme seu ambiente Kaggle)
# =============================================================

# --- Diretórios de dados (formato ImageFolder: uma subpasta por classe) ---
TRAIN_DIR = '/kaggle/input/datasets/arnabkumarroy02/ferplus/train'
VAL_DIR   = '/kaggle/input/datasets/arnabkumarroy02/ferplus/validation'

# --- Diretórios de saída (persistem durante a sessão do Kaggle e viram
#     "Output" do notebook, podendo ser reaproveitados em uma nova sessão) ---
WORK_DIR  = '/kaggle/working'
CKPT_DIR  = os.path.join(WORK_DIR, 'checkpoints')
STUDY_DB  = os.path.join(WORK_DIR, 'optuna_study.db')

os.makedirs(CKPT_DIR, exist_ok=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Dispositivo: {DEVICE}")

# --- Configuração da busca de hiperparâmetros ---
N_TRIALS       = 200    # nº de combinações de hiperparâmetros a testar
SEARCH_EPOCHS  = 8      # épocas curtas por trial (só para comparar configs)
SEARCH_TIMEOUT = None   # em segundos, ex. 6*3600 para limitar a 6h; None = sem limite de tempo

# --- Configuração do treino final (com os melhores hiperparâmetros) ---
FINAL_EPOCHS = 50

NUM_CLASSES = 8
CLASSES = ['Raiva', 'Nojo', 'Medo', 'Felicidade',
           'Tristeza', 'Surpresa', 'Neutro', 'Desprezo']


Dispositivo: cuda


In [3]:
# =============================================================
# 1. BLOCO RESIDUAL COM GARGALO (ResBottleneckBlock)
# =============================================================

class ResBottleneckBlock(nn.Module):
    """
    Bloco residual com gargalo (bottleneck), núcleo da AnyNet.
    Parâmetros:
        in_channels  : canais de entrada
        out_channels : canais de saída
        stride       : passo da convolução principal
        group_width  : largura dos grupos (convoluções agrupadas)
        bottleneck_ratio: razão de redução no gargalo
    """
    def __init__(self, in_channels, out_channels,
                 stride=1, group_width=1, bottleneck_ratio=1):
        super().__init__()

        mid_channels_raw = max(group_width, int(out_channels * bottleneck_ratio))
        num_groups   = max(1, round(mid_channels_raw / group_width))
        mid_channels = num_groups * group_width

        self.conv_block = nn.Sequential(
            nn.Conv2d(in_channels, mid_channels,
                      kernel_size=1, bias=False),
            nn.BatchNorm2d(mid_channels),
            nn.ReLU(inplace=True),

            nn.Conv2d(mid_channels, mid_channels,
                      kernel_size=3, stride=stride,
                      padding=1, groups=num_groups, bias=False),
            nn.BatchNorm2d(mid_channels),
            nn.ReLU(inplace=True),

            nn.Conv2d(mid_channels, out_channels,
                      kernel_size=1, bias=False),
            nn.BatchNorm2d(out_channels),
        )

        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels,
                          kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels),
            )

        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        return self.relu(self.conv_block(x) + self.shortcut(x))


# =============================================================
# 2. ESTÁGIO DA ANYNET
# =============================================================

class AnyNetStage(nn.Module):
    """
    Um estágio completo da AnyNet: sequência de d blocos residuais.
    O primeiro bloco pode reduzir a resolução espacial (stride=2).
    """
    def __init__(self, in_channels, out_channels,
                 depth, stride=2,
                 group_width=1, bottleneck_ratio=1):
        super().__init__()

        layers = []
        for i in range(depth):
            layers.append(
                ResBottleneckBlock(
                    in_channels  = in_channels if i == 0 else out_channels,
                    out_channels = out_channels,
                    stride       = stride if i == 0 else 1,
                    group_width  = group_width,
                    bottleneck_ratio = bottleneck_ratio,
                )
            )
        self.stage = nn.Sequential(*layers)

    def forward(self, x):
        return self.stage(x)


# =============================================================
# 3. MODELO ANYNET COMPLETO
# =============================================================

class AnyNetFER(nn.Module):
    """
    AnyNet adaptada para Reconhecimento de Expressões Faciais.

    Entrada : imagem 1×48×48 (escala de cinza, FER2013/FERPlus)
    Saída   : logits para NUM_CLASSES classes de expressão
    """
    def __init__(self,
                 num_classes   = NUM_CLASSES,
                 stem_width    = 32,
                 depths        = [1, 2, 4, 2],
                 widths        = [32, 64, 128, 256],
                 group_width   = 16,
                 bottleneck_ratio = 0.5,
                 dropout_rate  = 0.4):
        super().__init__()

        self.stem = nn.Sequential(
            nn.Conv2d(1, stem_width,
                      kernel_size=3, stride=2,
                      padding=1, bias=False),
            nn.BatchNorm2d(stem_width),
            nn.ReLU(inplace=True),
        )

        self.stages = nn.ModuleList()
        in_ch = stem_width
        for d, w, s in zip(depths, widths, [1, 2, 2, 2]):
            self.stages.append(
                AnyNetStage(
                    in_channels      = in_ch,
                    out_channels     = w,
                    depth            = d,
                    stride           = s,
                    group_width      = group_width,
                    bottleneck_ratio = bottleneck_ratio,
                )
            )
            in_ch = w

        self.head = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Dropout(dropout_rate),
            nn.Linear(widths[-1], num_classes),
        )

        self._initialize_weights()

    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(
                    m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, std=0.01)
                nn.init.zeros_(m.bias)

    def forward(self, x):
        x = self.stem(x)
        for stage in self.stages:
            x = stage(x)
        return self.head(x)


In [4]:
# =============================================================
# 4. DATASET — FERPlus (formato ImageFolder)
# =============================================================
# CORREÇÃO: o notebook original não importava ImageFolder e passava um
# DataFrame para FERPlusDataset em vez de um caminho de diretório.
# Aqui o dataset espera um diretório organizado por classe:
#   root_dir/
#     Raiva/*.png
#     Nojo/*.png
#     ...

class FERPlusDataset(Dataset):
    """Carrega o FERPlus a partir de uma pasta organizada por classe (ImageFolder)."""
    def __init__(self, root_dir, transform=None):
        self.dataset = ImageFolder(root=root_dir, transform=transform)
        self.classes = self.dataset.classes

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        return self.dataset[idx]


def build_transforms():
    """Aumentação de dados para treino e pré-processamento para validação."""

    # CORREÇÃO: faltava converter a imagem (PIL) para escala de cinza / tensor
    # antes do Normalize, senão o Normalize falha ou opera sobre PIL.Image.
    train_transform = transforms.Compose([
        transforms.Grayscale(num_output_channels=1),
        transforms.Resize((48, 48)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomRotation(degrees=10),
        transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.5], std=[0.5]),
    ])

    val_transform = transforms.Compose([
        transforms.Grayscale(num_output_channels=1),
        transforms.Resize((48, 48)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.5], std=[0.5]),
    ])

    return train_transform, val_transform


In [5]:
# =============================================================
# 5. LOOPS DE TREINO E AVALIAÇÃO
# =============================================================

def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss, total_correct, total = 0.0, 0, 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss    = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        preds = outputs.argmax(dim=1)
        total_correct += (preds == labels).sum().item()
        total_loss    += loss.item() * images.size(0)
        total         += images.size(0)

    return total_loss / total, total_correct / total


def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, total_correct, total = 0.0, 0, 0

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss    = criterion(outputs, labels)

            preds = outputs.argmax(dim=1)
            total_correct += (preds == labels).sum().item()
            total_loss    += loss.item() * images.size(0)
            total         += images.size(0)

    return total_loss / total, total_correct / total


In [6]:
# =============================================================
# 6. UTILITÁRIOS DE CHECKPOINT
# =============================================================
# Pensado para o Kaggle: a sessão pode cair (limite de tempo de GPU,
# desconexão, restart manual). Salvamos tudo que é necessário para
# retomar exatamente de onde parou: pesos do modelo, estado do
# otimizador e do scheduler, época atual e melhor métrica até então.

def save_checkpoint(path, model, optimizer=None, scheduler=None,
                     epoch=None, best_val_acc=None, extra=None):
    state = {
        'model_state': model.state_dict(),
        'epoch': epoch,
        'best_val_acc': best_val_acc,
    }
    if optimizer is not None:
        state['optimizer_state'] = optimizer.state_dict()
    if scheduler is not None:
        state['scheduler_state'] = scheduler.state_dict()
    if extra is not None:
        state['extra'] = extra

    # grava em arquivo temporário e depois renomeia (evita checkpoint
    # corrompido caso a sessão caia no meio da escrita)
    tmp_path = path + '.tmp'
    torch.save(state, tmp_path)
    os.replace(tmp_path, path)


def load_checkpoint(path, model, optimizer=None, scheduler=None, device='cpu'):
    if not os.path.exists(path):
        return None
    ckpt = torch.load(path, map_location=device)
    model.load_state_dict(ckpt['model_state'])
    if optimizer is not None and 'optimizer_state' in ckpt:
        optimizer.load_state_dict(ckpt['optimizer_state'])
    if scheduler is not None and 'scheduler_state' in ckpt:
        scheduler.load_state_dict(ckpt['scheduler_state'])
    return ckpt


In [7]:
# =============================================================
# 7. FUNÇÃO OBJETIVO DO OPTUNA
# =============================================================
# Cada trial treina o modelo por poucas épocas (SEARCH_EPOCHS) com uma
# combinação de hiperparâmetros e retorna a melhor acurácia de validação
# obtida. Trials claramente ruins são interrompidos cedo (pruning).

def objective(trial):
    # --- espaço de busca ---
    lr               = trial.suggest_float('lr', 1e-5, 1e-2, log=True)
    weight_decay     = trial.suggest_float('weight_decay', 1e-6, 1e-2, log=True)
    batch_size       = trial.suggest_categorical('batch_size', [32, 64, 128])
    dropout_rate     = trial.suggest_float('dropout_rate', 0.2, 0.6)
    group_width      = trial.suggest_categorical('group_width', [8, 16, 32])
    bottleneck_ratio = trial.suggest_float('bottleneck_ratio', 0.25, 1.0)
    optimizer_name   = trial.suggest_categorical('optimizer', ['AdamW', 'SGD'])

    train_tf, val_tf = build_transforms()
    train_ds = FERPlusDataset(TRAIN_DIR, transform=train_tf)
    val_ds   = FERPlusDataset(VAL_DIR,   transform=val_tf)

    train_loader = DataLoader(train_ds, batch_size=batch_size,
                               shuffle=True, num_workers=2, pin_memory=True)
    val_loader   = DataLoader(val_ds, batch_size=batch_size,
                               shuffle=False, num_workers=2, pin_memory=True)

    model = AnyNetFER(
        num_classes=NUM_CLASSES,
        group_width=group_width,
        bottleneck_ratio=bottleneck_ratio,
        dropout_rate=dropout_rate,
    ).to(DEVICE)

    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

    if optimizer_name == 'AdamW':
        optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    else:
        optimizer = optim.SGD(model.parameters(), lr=lr, momentum=0.9,
                               weight_decay=weight_decay, nesterov=True)

    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=SEARCH_EPOCHS)

    best_val_acc = 0.0
    trial_ckpt_path = os.path.join(CKPT_DIR, f'trial_{trial.number}_best.pth')

    for epoch in range(1, SEARCH_EPOCHS + 1):
        train_epoch(model, train_loader, criterion, optimizer, DEVICE)
        _, vl_acc = evaluate(model, val_loader, criterion, DEVICE)
        scheduler.step()

        if vl_acc > best_val_acc:
            best_val_acc = vl_acc
            # checkpoint do melhor modelo *dentro* deste trial
            save_checkpoint(trial_ckpt_path, model, optimizer, scheduler,
                             epoch=epoch, best_val_acc=best_val_acc)

        # reporta ao Optuna para permitir pruning de trials ruins
        trial.report(vl_acc, epoch)
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

    return best_val_acc


In [ ]:
# =============================================================
# 8. EXECUÇÃO DO ESTUDO OPTUNA (retomável)
# =============================================================
# O estudo é salvo em SQLite (STUDY_DB). Se o kernel do Kaggle cair e
# você reiniciar o notebook (mantendo /kaggle/working, ex. reabrindo a
# mesma sessão ou anexando o output de uma versão anterior como input),
# `load_if_exists=True` faz o Optuna continuar de onde parou em vez de
# recomeçar a busca do zero.

sampler = optuna.samplers.TPESampler(seed=42)
pruner  = optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=3)

study = optuna.create_study(
    study_name='anynet_fer_optuna',
    direction='maximize',
    sampler=sampler,
    pruner=pruner,
    storage=f'sqlite:///{STUDY_DB}',
    load_if_exists=True,
)

trials_ja_feitos = len([t for t in study.trials if t.state in (TrialState.COMPLETE, TrialState.PRUNED)])
trials_restantes = max(0, N_TRIALS - trials_ja_feitos)
print(f"Trials já concluídos: {trials_ja_feitos} | Trials restantes: {trials_restantes}")

try:
    if trials_restantes > 0:
        study.optimize(objective, n_trials=trials_restantes,
                        timeout=SEARCH_TIMEOUT, gc_after_trial=True)
except KeyboardInterrupt:
    print("Busca interrompida manualmente — progresso já está salvo no SQLite.")

print("\nMelhor valor (acurácia de validação):", study.best_value)
print("Melhores hiperparâmetros encontrados:")
for k, v in study.best_params.items():
    print(f"  {k}: {v}")

best_params = study.best_params


[I 2026-07-06 18:30:18,087] Using an existing study with name 'anynet_fer_optuna' instead of creating a new one.


Trials já concluídos: 0 | Trials restantes: 200


[I 2026-07-06 18:44:28,305] Trial 4 finished with value: 0.18415058146505214 and parameters: {'lr': 0.0001329291894316216, 'weight_decay': 0.006351221010640704, 'batch_size': 32, 'dropout_rate': 0.26239780813448105, 'group_width': 16, 'bottleneck_ratio': 0.7810544333470342, 'optimizer': 'SGD'}. Best is trial 4 with value: 0.18415058146505214.
[I 2026-07-06 18:54:19,869] Trial 5 finished with value: 0.2418175278743556 and parameters: {'lr': 0.00314288089084011, 'weight_decay': 7.068974950624607e-06, 'batch_size': 128, 'dropout_rate': 0.4099025726528951, 'group_width': 32, 'bottleneck_ratio': 0.35462039548903135, 'optimizer': 'SGD'}. Best is trial 5 with value: 0.2418175278743556.
[I 2026-07-06 19:04:02,513] Trial 6 finished with value: 0.484714063061983 and parameters: {'lr': 0.00023345864076016249, 'weight_decay': 0.0013826232179369874, 'batch_size': 128, 'dropout_rate': 0.2185801650879991, 'group_width': 8, 'bottleneck_ratio': 0.96166415294, 'optimizer': 'AdamW'}. Best is trial 6 with

In [ ]:
# =============================================================
# 8.1 GRÁFICO — EVOLUÇÃO DA ACURÁCIA AO LONGO DOS TRIALS
# =============================================================

import matplotlib.pyplot as plt

df = study.trials_dataframe()
df = df[df['state'] == 'COMPLETE'].sort_values('number')

trial_number = df['number']
accuracy     = df['value']

best_idx = accuracy.idxmax()

fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(trial_number, accuracy, color='#3B82F6', linewidth=1.5,
        marker='o', markersize=4, markerfacecolor='#3B82F6',
        markeredgewidth=0)

ax.scatter(df.loc[best_idx, 'number'], df.loc[best_idx, 'value'],
           s=80, color='#F59E0B', zorder=5,
           label=f"Melhor trial (#{int(df.loc[best_idx, 'number'])}: "
                 f"{df.loc[best_idx, 'value']:.4f})")

ax.set_xlim(0, 108)
ax.set_ylim(0.0, 0.7)
ax.set_xlabel('Trial')
ax.set_ylabel('Acurácia de validação')
ax.set_title('Evolução da acurácia ao longo dos trials (Optuna)')

ax.grid(True, axis='y', color='#E5E7EB', linewidth=0.8, zorder=0)
ax.set_axisbelow(True)
for spine in ('top', 'right'):
    ax.spines[spine].set_visible(False)

ax.legend(frameon=False, loc='lower right')

plt.tight_layout()
plt.show()


In [ ]:
# =============================================================
# 9. TREINO FINAL COM OS MELHORES HIPERPARÂMETROS (retomável)
# =============================================================
# Usa mais épocas (FINAL_EPOCHS) que os trials de busca. Salva um
# checkpoint a cada época em CKPT_DIR/final_checkpoint.pth; se essa
# célula for reexecutada (por ex. após a sessão do Kaggle cair), o
# treino continua automaticamente da última época salva.

def train_final(best_params, epochs=FINAL_EPOCHS, resume=True):
    train_tf, val_tf = build_transforms()
    train_ds = FERPlusDataset(TRAIN_DIR, transform=train_tf)
    val_ds   = FERPlusDataset(VAL_DIR,   transform=val_tf)

    train_loader = DataLoader(train_ds, batch_size=best_params['batch_size'],
                               shuffle=True, num_workers=4, pin_memory=True)
    val_loader   = DataLoader(val_ds, batch_size=best_params['batch_size'],
                               shuffle=False, num_workers=4, pin_memory=True)

    model = AnyNetFER(
        num_classes=NUM_CLASSES,
        group_width=best_params['group_width'],
        bottleneck_ratio=best_params['bottleneck_ratio'],
        dropout_rate=best_params['dropout_rate'],
    ).to(DEVICE)

    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

    if best_params['optimizer'] == 'AdamW':
        optimizer = optim.AdamW(model.parameters(), lr=best_params['lr'],
                                 weight_decay=best_params['weight_decay'])
    else:
        optimizer = optim.SGD(model.parameters(), lr=best_params['lr'], momentum=0.9,
                               weight_decay=best_params['weight_decay'], nesterov=True)

    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    print(f"Parâmetros treináveis: "
          f"{sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

    ckpt_path      = os.path.join(CKPT_DIR, 'final_checkpoint.pth')
    best_ckpt_path = os.path.join(CKPT_DIR, 'anynet_fer_best.pth')

    start_epoch  = 1
    best_val_acc = 0.0

    if resume:
        ckpt = load_checkpoint(ckpt_path, model, optimizer, scheduler, DEVICE)
        if ckpt is not None:
            start_epoch  = ckpt['epoch'] + 1
            best_val_acc = ckpt.get('best_val_acc', 0.0)
            print(f"Checkpoint encontrado — retomando treino a partir da época {start_epoch} "
                  f"(melhor val_acc até então: {best_val_acc:.4f})")

    for epoch in range(start_epoch, epochs + 1):
        t0 = time.time()
        tr_loss, tr_acc = train_epoch(model, train_loader, criterion, optimizer, DEVICE)
        vl_loss, vl_acc = evaluate(model, val_loader, criterion, DEVICE)
        scheduler.step()
        dt = time.time() - t0

        print(f"Época {epoch:03d}/{epochs} | "
              f"Loss Treino: {tr_loss:.4f} | Acc Treino: {tr_acc:.4f} | "
              f"Loss Val: {vl_loss:.4f} | Acc Val: {vl_acc:.4f} | {dt:.1f}s")

        if vl_acc > best_val_acc:
            best_val_acc = vl_acc
            save_checkpoint(best_ckpt_path, model, optimizer, scheduler,
                             epoch=epoch, best_val_acc=best_val_acc)
            print(f"  -> Melhor modelo salvo (val_acc: {best_val_acc:.4f})")

        # checkpoint "de retomada" salvo TODA época, mesmo se não for a melhor,
        # para nunca perder mais que 1 época de progresso caso a sessão caia
        save_checkpoint(ckpt_path, model, optimizer, scheduler,
                         epoch=epoch, best_val_acc=best_val_acc)

    return model


final_model = train_final(best_params, epochs=FINAL_EPOCHS, resume=True)


In [ ]:
# =============================================================
# 10. INFERÊNCIA EM NOVA IMAGEM
# =============================================================

def predict(image_path, model_path=os.path.join(CKPT_DIR, 'anynet_fer_best.pth'),
            best_params=None):
    """
    Realiza inferência em uma imagem facial externa.
    Requer: OpenCV para captura e pré-processamento.
    """
    import cv2

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    kwargs = {}
    if best_params is not None:
        kwargs = dict(
            group_width=best_params['group_width'],
            bottleneck_ratio=best_params['bottleneck_ratio'],
            dropout_rate=best_params['dropout_rate'],
        )

    model = AnyNetFER(num_classes=NUM_CLASSES, **kwargs)
    ckpt = torch.load(model_path, map_location=device)
    model.load_state_dict(ckpt['model_state'] if 'model_state' in ckpt else ckpt)
    model.to(device).eval()

    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    img = cv2.resize(img, (48, 48))

    tensor = torch.tensor(img / 255.0, dtype=torch.float32)
    tensor = (tensor - 0.5) / 0.5
    tensor = tensor.unsqueeze(0).unsqueeze(0)     # [1, 1, 48, 48]
    tensor = tensor.to(device)

    with torch.no_grad():
        logits     = model(tensor)
        probs      = torch.softmax(logits, dim=1).squeeze()
        pred_class = probs.argmax().item()

    print(f"Expressão detectada: {CLASSES[pred_class]}")
    print("Probabilidades por classe:")
    for cls, prob in zip(CLASSES, probs.tolist()):
        bar = '█' * int(prob * 30)
        print(f"  {cls:<12}: {prob:.4f} {bar}")

    return CLASSES[pred_class], probs.tolist()
